# Причинный вывод на практике: Lalonde NSW
## Часть 4. DML

Часть 3 снизила смещение через matching/PSM/PSW. Здесь тот же датасет, но
эффект оценивается через AIPW-score с ML-моделями вместо одной
параметрической формы.

## Метод

DML: две вспомогательные модели - `g0(X)=E[Y|T=0,X]`,
`m(X)=P(T=1|X)` - и AIPW-score, устойчивый к ошибке первого порядка в вспомогательных моделях (orthogonality).

Cross-fitting (K=5): g0/m обучаются на K-1 фолдах, предсказываются на
оставшемся - иначе overfitting bias.

Оцениваем ATT, не ATE.

In [1]:
!pip install -q causaldata --break-system-packages 2>/dev/null || pip install -q causaldata

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 28.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
import sys
sys.path.append('.')

import numpy as np
import pandas as pd
from scipy import stats
from causaldata import nsw_mixtape, cps_mixtape
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import roc_auc_score

from ci_utils import load_benchmark, load_results, append_result

np.random.seed(42)

nsw = nsw_mixtape.load_pandas().data
cps = cps_mixtape.load_pandas().data
treated = nsw[nsw['treat'] == 1].copy()
control = cps.copy()
df = pd.concat([treated, control], ignore_index=True).reset_index(drop=True)

covariates = ['age', 'educ', 'black', 'hisp', 'marr', 'nodegree', 're74', 're75']
X = df[covariates].values
T = df['treat'].values
Y = df['re78'].values
n = len(df)
print(f'Датасет: {n} наблюдений ({T.sum()} treated)')

Датасет: 16177 наблюдений (185 treated)


## 1. Cross-fitting: обучение nuisance-моделей

In [3]:
K = 5
kf = KFold(n_splits=K, shuffle=True, random_state=42)

g0_pred = np.zeros(n)
m_pred = np.zeros(n)

for train_idx, test_idx in kf.split(X):
    Xtr, Ttr, Ytr = X[train_idx], T[train_idx], Y[train_idx]
    Xte = X[test_idx]

    clf = RandomForestClassifier(n_estimators=300, max_depth=5, min_samples_leaf=20, random_state=42)
    clf.fit(Xtr, Ttr)
    m_pred[test_idx] = clf.predict_proba(Xte)[:, 1]

    reg0 = RandomForestRegressor(n_estimators=300, max_depth=4, min_samples_leaf=10, random_state=42)
    reg0.fit(Xtr[Ttr == 0], Ytr[Ttr == 0])
    g0_pred[test_idx] = reg0.predict(Xte)

print(f'Propensity AUC (out-of-fold): {roc_auc_score(T, m_pred):.3f}')
print(f'Propensity range: [{m_pred.min():.3f}, {m_pred.max():.3f}]')

Propensity AUC (out-of-fold): 0.973
Propensity range: [0.000, 0.547]


AUC 0.97 (out-of-fold) - прямое следствие дисбаланса ковариат, не
переобучение.

## 2. AIPW-score и оценка ATT

In [4]:
m_clip = np.clip(m_pred, 0, 0.99)

psi = T * (Y - g0_pred) - (1 - T) * (m_clip / (1 - m_clip)) * (Y - g0_pred)

p = T.mean()
att = psi.sum() / T.sum()

var = np.mean((psi - att * T) ** 2) / p ** 2
se_att = np.sqrt(var / n)
ci_att = (att - 1.96 * se_att, att + 1.96 * se_att)
z = att / se_att
pval = 2 * (1 - stats.norm.cdf(abs(z)))

print(f'DML ATT: {att:,.0f}')
print(f'SE: {se_att:,.0f}')
print(f'95% CI: [{ci_att[0]:,.0f}, {ci_att[1]:,.0f}]')
print(f'p-value: {pval:.4f}')

DML ATT: 917
SE: 651
95% CI: [-360, 2,193]
p-value: 0.1593


## 3. Сравнение с эталоном и PSM

In [5]:
benchmark = load_benchmark('benchmark.json')
history = load_results('results.csv')

rows = [('Эталон', benchmark['ate'], 0.0)]
for label in ['Naive diff-in-means (NSW+CPS)', 'Matching (Mahalanobis, NN)',
              'PSM (nearest-neighbor, caliper)', 'PSW (Hajek IPW, ATT weights)']:
    v = history.loc[label, 'ate']
    rows.append((label, v, v - benchmark['ate']))
rows.append(('DML ATT', att, att - benchmark['ate']))

for name, val, bias in rows:
    print(f'{name:34s} {val:>10,.0f}   bias {bias:>9,.0f}')

Эталон                                  1,794   bias         0
Naive diff-in-means (NSW+CPS)          -8,498   bias   -10,292
Matching (Mahalanobis, NN)              2,089   bias       294
PSM (nearest-neighbor, caliper)         1,663   bias      -131
PSW (Hajek IPW, ATT weights)            1,118   bias      -677
DML ATT                                   917   bias      -878


DML даёт наибольшее по модулю смещение среди методов коррекции (Matching,
PSM, PSW) - при этом SE (651) не хуже, а даже чуть ниже, чем у Matching/PSM:
проблема не в разбросе (precision), а в том, что точечная оценка систематически
дальше от истины (bias). Ожидаемо при малой treated-группе (185) и сильном
дисбалансе: гибкость ML-моделей не успевает окупиться на таком объёме данных.

In [6]:
result = {
    'ate': float(att),
    'se': float(se_att),
    'ci_low': float(ci_att[0]),
    'ci_high': float(ci_att[1]),
    'p_value': float(pval),
    'n_treat': int(T.sum()),
    'n_control': int((T == 0).sum()),
}
append_result(result, label='DML (AIPW, ATT, RF cross-fit)', path='results.csv')

,method,ate,se,ci_low,ci_high,p_value,n_treat,n_control
0,Naive diff-in-means (NSW+CPS),-8497.515625,712.020724,-9893.155036,-7101.876214,1.074814e-32,185,15992
1,"Matching (Mahalanobis, NN)",2088.650204,721.675624,664.827106,3512.473301,4.261236e-03,185,124
2,"PSM (nearest-neighbor, caliper)",1662.849182,738.834401,205.172839,3120.525525,2.559200e-02,185,131
3,"PSW (Hajek IPW, ATT weights)",1117.783535,640.877282,-115.226081,2357.479983,NaN,185,15992
4,"DML (AIPW, ATT, RF cross-fit)",916.772265,651.302888,-359.781395,2193.325926,1.592503e-01,185,15992


## Что дальше

Часть 5 - валидация: placebo-тест (случайный "фейковый" treatment, эффект должен
быть ≈0) и проверка чувствительности к набору confounders для PSM и DML.